# Кандидатогенерация для категории «Услуги» Авито

**Задача.** Для каждого из 2 452 поисковых запросов отобрать из корпуса (189 212 объявлений) до 50 кандидатов так,
чтобы в них попало объявление, которое пользователь в итоге выбрал. Метрика — **Recall@50**.

**Решение в одном абзаце.** Сначала корпус сужается до «разрешённых» локаций. Их три вида: своя, те, куда
реально уходят пользователи из этой локации (по train), и соседние в радиусе 60 км. Затем несколько независимых
источников дают кандидатов: BM25 по заголовку и полному тексту, символьные n-граммы, расширение запроса через
похожие запросы из train, классификатор подкатегории и небольшой глобальный источник без фильтра по локации.
Получается пул ~400 объявлений на запрос с полнотой ~0.96. Из пула LightGBM LambdaRank по 30 признакам
выбирает 50.

**Как устроен ноутбук.** Логика лежит в модулях `src/`, ноутбук вызывает их по шагам и показывает результаты.
Каждый этап кэшируется в `cache/`: повторный запуск продолжает с первого незавершённого этапа.
Тот же пайплайн из консоли: `python run_pipeline.py --data_dir <папка с данными>`.

| Модуль | Что внутри |
|---|---|
| `src/textproc.py` | нормализация и лемматизация (pymorphy3) |
| `src/params.py` | разбор фильтров «Вид услуги / Тип услуги / Рейтинг» |
| `src/data.py` | потоковая подготовка данных с кэшем по кускам |
| `src/split.py` | схема валидации, имитирующая бенчмарк |
| `src/retriever.py` | источники кандидатов и признаки |
| `src/ranker.py` | LightGBM LambdaRank, top-k, Recall@K |
| `src/pipeline.py` | этапы решения с кэшированием |

In [ ]:
import sys, json
sys.path.insert(0, 'src')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pipeline import Pipeline

DATA_DIR = 'data'      # папка с train.parquet, benchmark_items.parquet, benchmark_queries.parquet
CACHE_DIR = 'cache'
pd.set_option('display.max_colwidth', 80)
pl = Pipeline(DATA_DIR, CACHE_DIR, n_holdout=10_000, train_rows=0)   # train_rows=0 -> весь train

## 1. Подготовка данных

`prepare()` объединяет объявления корпуса и объявления из train (их тексты нужны для расширения запроса и
валидации). Затем лемматизирует заголовок, параметры и первые 120 слов описания и разбирает фильтры.
Train читается **кусками по ~20 тыс. строк**, и каждый кусок сразу сохраняется. Так пиковая память
не зависит от размера train, а после обрыва подготовка продолжается с первого необработанного куска.

In [ ]:
pl.prep()
items, pairs, bq = pl.items, pl.pairs, pl.bq
items.head(3)

## 2. Что показал анализ данных
### 2.1 Локация — главный фильтр

In [ ]:
p = pairs.merge(items[['item_id', 'item_location_id']], on='item_id')
same = (p.search_location_id == p.item_location_id)
print(f'локация объявления == локация поиска: {same.mean():.1%}')
mis = p[~same]
print(f'среди несовпадений локация поиска не встречается у объявлений (регион / вся страна): '
      f'{(~mis.search_location_id.isin(set(items.item_location_id))).mean():.1%}')
for loc in mis.search_location_id.value_counts().index[:3]:
    x = p[p.search_location_id == loc].item_location_id.value_counts(normalize=True)
    print(f'  поиск в {loc}: выборы в {len(x)} локациях, топ-3 доли {x.head(3).round(3).to_dict()}')

**Вывод.** Фильтровать нужно не только по своей локации, но и по выученному распределению
P(локация объявления | локация поиска), плюс учитывать соседние города. Без этого теряется ~8% релевантных
объявлений.

### 2.2 Запомнить объявления из train не получится

In [ ]:
bench = items[items.in_bench]
print(f'объявлений корпуса, встречавшихся в train: {bench.item_id.isin(set(pairs.item_id)).mean():.1%}')
print(f'текстов запросов бенчмарка, встречавшихся в train: {bq.search_query.isin(set(pairs.search_query)).mean():.1%}')

**Вывод.** Нужно сопоставлять смысл текста. Из train при этом можно выучить обобщаемые связи: запрос →
подкатегория, запрос → слова в заголовках выбранных объявлений, локация → локации.

### 2.3 Бенчмарк не похож на случайную выборку из train

In [ ]:
g = pairs.drop_duplicates(['search_query', 'search_location_id', 'search_infm_params_text'])
u = g.drop_duplicates('search_query')
pd.DataFrame({
    'train (все группы)': [g.search_query.str.split().str.len().mean(), (g.search_infm_params_text != '').mean()],
    'train (уникальные тексты)': [u.search_query.str.split().str.len().mean(), (u.search_infm_params_text != '').mean()],
    'бенчмарк': [bq.search_query.str.split().str.len().mean(), (bq.search_infm_params_text != '').mean()],
}, index=['слов в запросе', 'доля с фильтрами']).round(2)

**Вывод.** Бенчмарк ближе к выборке уникальных текстов (длинный хвост), и валидацию нужно строить так же.

### 2.4 Фильтры из параметров запроса надёжны

In [ ]:
p = pairs.merge(items[['item_id', 'i_vid', 'i_tips', 'item_rating']], on='item_id')
v, t, r = p[p.q_vid != ''], p[p.q_tip != ''], p[p.q_rating4 == 1]
print(f'«Вид услуги» совпадает: {(v.q_vid == v.i_vid).mean():.1%} (n={len(v)})')
print(f'«Тип услуги» есть у объявления: {np.mean([a in b.split("|") for a, b in zip(t.q_tip, t.i_tips)]):.1%} (n={len(t)})')
print(f'«Рейтинг 4+» соблюдён: {(r.item_rating.fillna(0) >= 4).mean():.1%} (n={len(r)})')

## 3. Валидация, имитирующая бенчмарк (`src/split.py`)

1. Отложенные запросы берутся **равномерно по уникальным текстам**: 10 000 текстов, по одной группе
   (текст, локация, фильтры) на текст.
2. Для 73% из них из обучения удаляются все строки с этим текстом (новый запрос), для 27% — только сама группа.
   Это доля знакомых текстов в бенчмарке.
3. Корпус валидации = корпус бенчмарка + релевантные объявления отложенных запросов.
4. Отложенные запросы делятся на обучение ранкера (2/3) и валидацию (1/3). Ранняя остановка делается на части
   обучающих запросов, валидация в обучении не участвует.

Все статистики (классификатор, переходы локаций, расширение запроса) учатся только на оставшемся train,
поэтому утечки нет.

## 4. Кандидатогенерация (`src/retriever.py`)

| Источник | Что ловит | Топ-N |
|---|---|---|
| BM25 по заголовку | точные совпадения слов | 150 |
| BM25 по заголовку ×2 + параметрам + описанию | слова в описании услуги | 150 |
| TF-IDF по символьным 3–4-граммам | опечатки, словоформы | 100 |
| расширение запроса через похожие запросы train | синонимы, жаргон | 150 |
| P(подкатегория \| запрос), SGD-логрег | «наращивание ногтей» → маникюр | 100 |
| «память» по похожим прошлым запросам | популярные объявления | 30 |
| глобально (весь корпус): BM25 + расширение | соседние города, «вся Россия» | 60 + 40 |

In [ ]:
pl.val_candidates()
pl.cand.head()

## 5. Ранкер и результаты

LightGBM LambdaRank оптимизирует порядок внутри каждого запроса (`group`). `lambdarank_truncation_level=60`
фокусирует обучение на границе топ-50. У модели 30 признаков: скоры источников (сырые и нормированные на
максимум в запросе), подкатегория, локация, расстояние, совпадение фильтров, рейтинг, популярность.

In [ ]:
pl.val_ranker()
rep = pl.report
pd.Series({'BM25 по всему корпусу': rep['bm25_whole_corpus'],
           'BM25 + фильтр по локации': rep['bm25_location_filter'],
           'Полнота пула (верхняя граница)': rep['candidate_pool_upper_bound'],
           'Итог: пул + LightGBM': rep['final_recall@50']}).round(4).to_frame('Recall@50')

In [ ]:
imp = pd.Series(rep['feature_importance']).head(15)
imp[::-1].plot.barh(figsize=(6, 5), title='Важность признаков (gain)'); plt.tight_layout()

## 6. Разбор ошибок на валидации

In [ ]:
va = pd.read_parquet(pl.f['val_scored'])
hq = pl.hq
titles = items.set_index('item_id').item_title_raw
va['rank'] = va.groupby('qi').score.rank(ascending=False, method='first')
rows = [(i, it) for i in hq.index[hq.part == 'val'] for it in hq.relevant[i]]
d = pd.DataFrame(rows, columns=['qi', 'item_id']).merge(va[['qi', 'item_id', 'rank']], how='left')
d['status'] = np.select([d['rank'].isna(), d['rank'] <= 50], ['нет в пуле', 'в топ-50'], 'в пуле, ниже 50')
d.status.value_counts(normalize=True).round(3)

In [ ]:
def show(status, n=8, seed=0):
    out = []
    for _, r in d[d.status == status].sample(n, random_state=seed).iterrows():
        top1 = va[(va.qi == r.qi) & (va['rank'] == 1)].item_id
        out.append({'запрос': hq.search_query[r.qi], 'выбранное': titles[r.item_id][:55],
                    'ранг': r['rank'], 'наш топ-1': titles[top1.iloc[0]][:45] if len(top1) else None})
    return pd.DataFrame(out)
show('в пуле, ниже 50')

Обычно здесь общий запрос («няня», «покраска волос»), и подходящих объявлений в городе больше 50. Наш топ-1 тоже релевантен: это неоднозначность, а не ошибка модели.

In [ ]:
show('нет в пуле')

Промахи мимо пула бывают трёх видов: у выбранного объявления общий заголовок без слов запроса («Мастер на час»), транслитерация («джисиби» → «JCB»), шум в разметке.

In [ ]:
vq = hq[hq.part == 'val']
hit = d.assign(h=d.status == 'в топ-50').groupby('qi').h.mean()
vq = vq.assign(recall=hit.reindex(vq.index).values, words=vq.search_query.str.split().str.len().clip(upper=5))
vq.groupby('words').recall.agg(['mean', 'size']).round(3)

## 7. Ответ для бенчмарка

Ранкер переобучается на всех отложенных запросах, а для запросов бенчмарка статистики (классификатор,
переходы локаций, расширение) учатся на **всём** train. Корпус — только `benchmark_items`.
`check_answer` проверяет все требования к файлу из условия.

In [ ]:
pl.final_ranker()
del pl.cand
pl.bench_candidates()
ans = pl.answer('answer.csv')
print(ans.shape, ans.answer.str.split().str.len().agg(['min', 'max']).to_dict())
ans.head()

In [ ]:
# глазами: что мы нашли для нескольких запросов бенчмарка
for i in [0, 5, 500, 1500]:
    ids = ans.answer[i].split()[:3]
    print(bq.search_query[i], '->', [titles[x][:40] for x in ids])